In [137]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root added:", project_root)

Project root added: /home/junix/marketing-content-verfication


In [138]:
from src.ingestion.load_products import load_products

df = load_products(
    "../data/raw/products.csv"
)

[INFO] Loaded 100 products.


In [139]:
from src.ingestion.load_products import load_products

from src.ingestion.preprocess import (
    preprocess_dataframe
)

df = load_products(
    "../data/raw/products.csv"
)

df = preprocess_dataframe(df)

[INFO] Loaded 100 products.


In [140]:
# from pathlib import Path

# print(Path.cwd())

In [141]:
# import os

# print(os.listdir())

In [142]:
print(df.columns.tolist())

['product_id', 'product_name', 'category', 'brand', 'source', 'price_inr', 'product_description', 'specifications_and_warranty', 'source_url']


In [143]:
import importlib

import src.ingestion.chunking as chunking

importlib.reload(chunking)

<module 'src.ingestion.chunking' from '/home/junix/marketing-content-verfication/src/ingestion/chunking.py'>

In [144]:
from src.ingestion.chunking import (
    create_chunks
)

In [145]:
product_chunks = create_chunks(
    df,
    strategy="product"
)

print(
    "Number of chunks:",
    len(product_chunks)
)

print(
    product_chunks[0]
)

[INFO] Created 100 chunks using 'product' strategy.
Number of chunks: 100
{'product_id': 'Elec-01', 'product_name': 'boAt Airdopes 141 Gen 2 TWS Earbuds', 'category': 'Electronics', 'chunk_type': 'product', 'text': 'Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds\n            Brand: boAt\n            Category: Electronics\n            Price: ₹999\n\n            Description:\n            True wireless earbuds with 48-hour total playback, 4-mic ENx tech for crystal-clear calls, Beast Mode low-latency gaming, and IPX4 sweat resistance.\n\n            Specifications & Warranty:\n            Drivers: 6mm | BT: v5.4 | Playback: 48 hrs (buds+case) | Fast Charge: 10 min = 180 min | IPX4 | Warranty: 1 Year'}


In [146]:
attribute_chunks = create_chunks(
    df,
    strategy="attribute"
)

print(
    "Number of chunks:",
    len(attribute_chunks)
)

print(
    attribute_chunks[0]
)

[INFO] Created 607 chunks using 'attribute' strategy.
Number of chunks: 607
{'product_id': 'Elec-01', 'product_name': 'boAt Airdopes 141 Gen 2 TWS Earbuds', 'category': 'Electronics', 'chunk_type': 'attribute', 'text': 'Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds\n                Brand: boAt\n                Category: Electronics\n\n                Drivers: 6mm'}


In [147]:
print(
    df["specifications_and_warranty"].iloc[0]
)

Drivers: 6mm | BT: v5.4 | Playback: 48 hrs (buds+case) | Fast Charge: 10 min = 180 min | IPX4 | Warranty: 1 Year


In [148]:
print("Product Chunks:", len(product_chunks))
print("Attribute Chunks:", len(attribute_chunks))

Product Chunks: 100
Attribute Chunks: 607


** Experiment 1 complete **

In [149]:
import time
import pandas as pd

from src.embeddings.embed_products import (
    embed_documents
)

In [150]:
import sys

print(sys.executable)

/home/junix/marketing-content-verfication/venv/bin/python


In [169]:
MODEL_NAME = "BAAI/bge-large-en-v1.5"

CHUNKING = "attribute"

TOP_K = 5

In [170]:
chunks = (
    attribute_chunks
    if CHUNKING == "attribute"
    else product_chunks
)

start = time.time()

embeddings = embed_documents(
    chunks,
    MODEL_NAME
)

embedding_time = time.time() - start

[INFO] Loading model: BAAI/bge-large-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

[INFO] Generated embeddings for 607 chunks.


In [171]:
print(MODEL_NAME)
print(CHUNKING)
print(TOP_K)
print(embeddings.shape)
print(embedding_time)

BAAI/bge-large-en-v1.5
attribute
5
(607, 1024)
453.39160323143005


In [172]:
from sklearn.metrics.pairwise import cosine_similarity

In [173]:
query = embeddings[0]

candidate = embeddings[1]

score = cosine_similarity(
    [query],
    [candidate]
)

print(score)

[[0.9642459]]


In [174]:
print(
    "Embedding Shape:",
    embeddings.shape
)

Embedding Shape: (607, 1024)


In [175]:
from src.vectordb.create_faiss import (
    create_index,
    save_index,
    load_index
)

from src.vectordb.search_faiss import (
    search_index
)

In [176]:
index = create_index(
    embeddings
)

[INFO] Added 607 vectors to FAISS.


In [177]:
save_index(
    index,
    "../models/faiss/products.index"
)

[INFO] Saved index: ../models/faiss/products.index


In [178]:
index = load_index(
    "../models/faiss/products.index"
)

[INFO] Loaded index: ../models/faiss/products.index


In [179]:
query_embedding = embeddings[0].reshape(1, -1)

distances, indices = search_index(
    index,
    query_embedding,
    k=5
)

print(indices)
print(distances)

[[0 1 4 2 5]]
[[1.0000001  0.9642459  0.96147954 0.91834915 0.8954707 ]]


In [180]:
for idx in indices[0]:
    print(chunks[idx]["text"])
    print("-" * 50)

Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds
                Brand: boAt
                Category: Electronics

                Drivers: 6mm
--------------------------------------------------
Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds
                Brand: boAt
                Category: Electronics

                BT: v5.4
--------------------------------------------------
Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds
                Brand: boAt
                Category: Electronics

                IPX4
--------------------------------------------------
Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds
                Brand: boAt
                Category: Electronics

                Playback: 48 hrs (buds+case)
--------------------------------------------------
Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds
                Brand: boAt
                Category: Electronics

                Warranty: 1 Year
--------------------------------------------------


In [181]:
import pandas as pd

try:
    experiment_df
except NameError:
    experiment_df = pd.DataFrame()

In [182]:
# experiment_df = pd.DataFrame()

In [183]:
current_result = pd.DataFrame({
    "Embedding Model": [MODEL_NAME],
    "Chunking": [CHUNKING],
    "Embedding Dimension": [embeddings.shape[1]],
    "Chunks": [len(chunks)],
    "Embedding Time (s)": [embedding_time],
    "Top-K": [TOP_K],
    # "Average Similarity": [distances.mean()],
    # "Maximum Similarity": [distances.max()]
})

In [184]:
# experiment_df = experiment_df.drop(
#     columns=[
#         "Average Similarity",
#         "Maximum Similarity"
#     ]
# )

In [185]:
experiment_df = pd.concat(
    [experiment_df, current_result],
    ignore_index=True
)

In [186]:
experiment_df

,Embedding Model,Chunking,Embedding Dimension,Chunks,Embedding Time (s),Top-K
0,BAAI/bge-small-en-v1.5,attribute,384,607,22.353668,5
1,BAAI/bge-small-en-v1.5,product,384,100,17.842308,5
2,BAAI/bge-large-en-v1.5,attribute,1024,607,453.391603,5
